# state_head_fold_consensus_target_v1

Purpose: generate fold-consensus rebound state-head target rows before any auxiliary PPO handoff.

Governance:
- Baseline: `state_head_fold_stability_guard_v1`.
- Role: diagnose / target generation guard.
- Changed layer: model-selection target generation only; no runtime execution change.
- Target gate remains `+30pp` relative return and `<=15%` MaxDD after future training.
- Accepted trade-off: if fold-consensus removes all non-trivial target lifts, block PPO and move back to label/factor discovery.

External references:
- Qlib reproducible quant workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validation/test trading workflow: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Probability of Backtest Overfitting / selection discipline: https://doi.org/10.21314/JCF.2016.322
- Supervised contrastive representation learning: https://arxiv.org/abs/2004.11362

Closest success:
- `rebound_representation_separability_v1` showed a compact `momentum_drawdown` state representation can separate rebound-like rows.

Closest failure:
- `state_head_fold_stability_guard_v1` showed every non-trivial target lift came from fold-label-unstable dates. This run is not a duplicate because it produces the actual consensus target candidate table and decides whether any target survives.


In [1]:

from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError(f'Cannot locate repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadConsensusTargetConfig,
    StateHeadLiftStabilityConfig,
    StateHeadTargetRecipeConfig,
    derive_state_head_consensus_targets,
    score_state_head_lift_stability,
    score_state_head_target_recipes,
)
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'state_head_fold_consensus_target_v1'
OUT_DIR = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'state_objective_fold_native_teacher_router_v1' / 'fold_native_teacher_targets.csv'
STABILITY_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'state_head_fold_stability_guard_v1' / 'selection.json'
LIFT_GUARD_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_head_lift_guard_v1' / 'selection.json'

with STABILITY_SELECTION.open() as fh:
    stability_selection = json.load(fh)
with LIFT_GUARD_SELECTION.open() as fh:
    lift_guard_selection = json.load(fh)

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
targets['fold'] = targets['fold'].astype(str)
targets['is_rebound_cluster'] = targets['is_rebound_cluster'].fillna(False).astype(bool)
targets['is_downtrend_state'] = targets['is_downtrend_state'].fillna(False).astype(bool)
targets['true_target_position'] = [state_objective_target_position_from_row(row) for row in targets.to_dict('records')]
targets['default_target_position'] = np.where(
    targets['is_downtrend_state'],
    pd.to_numeric(targets['risk_target_position'], errors='coerce'),
    pd.to_numeric(targets['teacher_position'], errors='coerce'),
)
base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_targets': str(TARGETS_PATH.relative_to(REPO_ROOT)),
    'source_stability_guard': str(STABILITY_SELECTION.relative_to(REPO_ROOT)),
    'rows': int(len(targets)),
    'unique_dates': int(targets['datetime'].nunique()),
    'folds': sorted(targets['fold'].unique().tolist()),
    'rebound_rows': int(targets['is_rebound_cluster'].sum()),
    'baseline_feature_set': lift_guard_selection['best_feature_set'],
    'baseline_threshold': float(lift_guard_selection['best_threshold']),
    'stability_guard_decision': stability_selection['decision'],
}
print(json.dumps(base_context, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "baseline_feature_set": "momentum_drawdown",
  "baseline_threshold": 0.5,
  "created_at": "2026-05-25T14:34:24.364552+00:00",
  "experiment": "state_head_fold_consensus_target_v1",
  "folds": [
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "rebound_rows": 7,
  "rows": 1261,
  "source_stability_guard": "runtime/model_selection/card/state_head_fold_stability_guard_v1/selection.json",
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv",
  "stability_guard_decision": "state_head_lift_fold_unstable_no_ppo_handoff",
  "unique_dates": 756
}


In [2]:

recipe_config = StateHeadTargetRecipeConfig(
    min_recall=0.70,
    min_precision=0.05,
    max_false_positive_rate=0.20,
    min_effective_lift=0.01,
    max_effective_false_positive_rate=0.01,
    max_false_positive_lift_mean=0.01,
    max_false_positive_lift_max=0.20,
    max_target_mae=0.02,
    max_rebound_mae=0.03,
    min_covered_folds=2,
)
recipe_result = score_state_head_target_recipes(
    targets,
    label_col='is_rebound_cluster',
    fold_col='fold',
    true_target_col='true_target_position',
    default_target_col='default_target_position',
    state_target_col='rebound_target_position',
    feature_sets={'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63']},
    thresholds=[float(lift_guard_selection['best_threshold'])],
    min_positive_train=2,
    min_positive_eval=2,
    config=recipe_config,
)
row_scores = recipe_result.row_scores.merge(
    targets.reset_index().rename(columns={'index': 'row_index'})[
        ['row_index', 'datetime', 'is_downtrend_state', 'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21']
    ],
    on='row_index',
    how='left',
)
stability = score_state_head_lift_stability(
    row_scores,
    group_col='datetime',
    label_col='is_rebound_cluster',
    config=StateHeadLiftStabilityConfig(min_lift=0.01, max_unstable_effective_lift_rate=0.0),
)
consensus = derive_state_head_consensus_targets(
    row_scores,
    group_col='datetime',
    label_col='is_rebound_cluster',
    default_target_col='default_target_position',
    proposed_target_col='proposed_target',
    target_lift_col='target_lift',
    predicted_col='predicted_state',
    config=StateHeadConsensusTargetConfig(min_sources=2, min_lift=0.01, min_positive_votes=1, max_negative_votes=0),
)

consensus.group_metrics.to_csv(OUT_DIR / 'consensus_group_metrics.csv', index=False)
consensus.consensus_targets.to_csv(OUT_DIR / 'consensus_targets.csv', index=False)
row_scores[row_scores['target_lift'].abs() > 0.01].to_csv(OUT_DIR / 'nonzero_lift_source_rows.csv', index=False)

print('recipe summary')
print(json.dumps(recipe_result.summary, indent=2, sort_keys=True))
print('stability summary')
print(json.dumps(stability.summary, indent=2, sort_keys=True))
print('consensus summary')
print(json.dumps(consensus.summary, indent=2, sort_keys=True))
print('consensus targets')
display(consensus.consensus_targets)
print('top conflict/effective groups')
display(consensus.group_metrics.head(12))


recipe summary
{
  "best_effective_false_positive_rate": 0.0007974481658692185,
  "best_false_positive_lift_max": 0.17985462658059442,
  "best_false_positive_lift_mean": 0.001462232736427597,
  "best_false_positive_rate": 0.09808612440191387,
  "best_feature_set": "momentum_drawdown",
  "best_rebound_target_mae": 0.028571428571428574,
  "best_state_precision": 0.05384615384615385,
  "best_state_recall": 1.0,
  "best_target_mae": 0.00030123285216541985,
  "best_threshold": 0.5,
  "recipe_gate": true
}
stability summary
{
  "effective_lift_rows": 5,
  "false_positive_effective_lift_rows": 1,
  "groups": 756,
  "positive_effective_lift_rows": 4,
  "rows": 1261,
  "stability_gate": false,
  "unstable_effective_lift_rate": 1.0,
  "unstable_effective_lift_rows": 5,
  "unstable_false_positive_effective_lift_rows": 1,
  "unstable_groups": 7
}
consensus summary
{
  "conflict_groups": 7,
  "consensus_gate": false,
  "consensus_target_groups": 0,
  "effective_lift_groups": 4,
  "groups": 756,
  "

,datetime,source_count,positive_labels,negative_labels,consensus_default_position,consensus_target_position,consensus_lift


top conflict/effective groups


,datetime,source_count,positive_labels,negative_labels,label_unstable,consensus_label,effective_lift_rows,positive_effective_lift_rows,false_positive_effective_lift_rows,consensus_effective_lift,consensus_default_position,consensus_target_position,consensus_lift
0,2020-06-03,2,1,1,True,False,2,1,1,False,0.250000,0.250000,0.0
1,2020-06-04,2,1,1,True,False,0,0,0,False,0.439927,0.439927,0.0
2,2020-06-05,2,1,1,True,False,0,0,0,False,0.606245,0.606245,0.0
3,2020-06-08,2,1,1,True,False,0,0,0,False,0.606245,0.606245,0.0
4,2021-04-05,2,1,1,True,False,1,1,0,False,0.243727,0.243727,0.0
5,2021-04-06,2,1,1,True,False,1,1,0,False,0.175599,0.175599,0.0
6,2021-04-07,2,1,1,True,False,1,1,0,False,0.247941,0.247941,0.0
7,2020-01-02,2,0,2,False,False,0,0,0,False,0.062500,0.062500,0.0
8,2020-01-03,2,0,2,False,False,0,0,0,False,0.062500,0.062500,0.0
9,2020-01-06,2,0,2,False,False,0,0,0,False,0.062500,0.062500,0.0


In [3]:

consensus_gate = bool(consensus.summary['consensus_gate'])
if consensus_gate:
    decision = 'fold_consensus_targets_ready_for_pre_ppo_handoff'
    next_mechanism = 'aux_rebound_state_head_ppo_with_drawdown_guard'
    training_allowed = True
    blocker = ''
else:
    decision = 'fold_consensus_removes_all_state_head_lifts_no_training'
    next_mechanism = 'discover_stable_rebound_labels_or_new_state_factor'
    training_allowed = False
    blocker = 'Fold-consensus generated zero non-trivial rebound target lifts; current state-head lift labels are too unstable for PPO.'

selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'source_experiment': 'state_head_fold_stability_guard_v1',
    'direct_comparison_baseline': 'rebound_state_head_lift_guard_v1',
    'decision': decision,
    'next_mechanism': next_mechanism,
    'training_allowed_from_consensus_targets': training_allowed,
    'promotion_allowed': False,
    'full_gate': False,
    'blocker': blocker,
    'recipe_gate': bool(recipe_result.summary['recipe_gate']),
    'stability_gate': bool(stability.summary['stability_gate']),
    'consensus_gate': consensus_gate,
    'source_effective_lift_rows': int(stability.summary['effective_lift_rows']),
    'unstable_effective_lift_rows': int(stability.summary['unstable_effective_lift_rows']),
    'consensus_target_groups': int(consensus.summary['consensus_target_groups']),
    'conflict_groups': int(consensus.summary['conflict_groups']),
    'acceptance_gate_for_next_run': {
        'stable_consensus_target_rows_required': True,
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'no_runtime_promotion_from_recipe_only': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://arxiv.org/abs/2004.11362',
    ],
    'sidecars': [
        'base_context.json',
        'consensus_group_metrics.csv',
        'consensus_targets.csv',
        'nonzero_lift_source_rows.csv',
        'summary.json',
    ],
}
summary = {
    'base_context': base_context,
    'recipe_summary': recipe_result.summary,
    'stability_summary': stability.summary,
    'consensus_summary': consensus.summary,
    'selection': selection,
    'conclusion': [
        f"Source effective target-lift rows: {selection['source_effective_lift_rows']}.",
        f"Fold-consensus target groups: {selection['consensus_target_groups']}.",
        f"Conflict groups: {selection['conflict_groups']}.",
        f"Decision: {decision}; next mechanism: {next_mechanism}.",
        'No runtime promotion; no PPO handoff until stable consensus labels or new factors create non-conflicting rebound target rows.',
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')
print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "fixed_split_required": true,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_recipe_only": true,
    "relative_return_min_pp": 30.0,
    "stable_consensus_target_rows_required": true,
    "walkforward_required": true
  },
  "blocker": "Fold-consensus generated zero non-trivial rebound target lifts; current state-head lift labels are too unstable for PPO.",
  "conflict_groups": 7,
  "consensus_gate": false,
  "consensus_target_groups": 0,
  "created_at": "2026-05-25T14:34:24.364552+00:00",
  "decision": "fold_consensus_removes_all_state_head_lifts_no_training",
  "direct_comparison_baseline": "rebound_state_head_lift_guard_v1",
  "experiment": "state_head_fold_consensus_target_v1",
  "external_sources": [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https:/

## Conclusion

This notebook is the single-run conclusion for fold-consensus target generation. If the consensus target table is empty, the state-head branch is not ready for auxiliary PPO despite the earlier representation and lift guards. The next useful work is stable rebound label discovery or a new state factor that survives fold consensus, not another PPO run on the current sparse lift labels.